# Full E1 Training: PhoWhisper-tiny LoRA on 26,671 Vietnamese Utterances

**Project:** Vietnamese Speech-to-Text Research (Phase 3 Full-Scale Experiment E1)  
**Target Environment:** Google Colab Linux GPU (NVIDIA Tesla T4 / L4 / A100)  
**Base Model:** `vinai/PhoWhisper-tiny` (Pinned commit `cc51d32be916efebde04ff549854fa1741cb5c02`)  
**PEFT Version:** `0.21.0`  
**LoRA Configuration:** $r=8, \alpha=16$, dropout=0.05, `target_modules=['q_proj', 'v_proj']`, `bias='none'` (147,456 trainable parameters, 0.389%)  
**Training Corpus:** 26,671 training-ready utterances (VIVOS train: 11,660 + ViMD train: 15,011)  
**Budget:** Micro Batch = 4, Grad Accum = 8, Effective Batch = 32, FP16 mixed precision  
**Schedule:** 834 steps/epoch × 3 epochs = **2,502 optimizer steps** (80,064 sample exposures)  
**Validation:** `manifests/full_val_manifest.csv` (1,900 ViMD valid utterances, 10.26 hours)  
**Frozen Gold Test:** `manifests/test_manifest.csv` (Evaluated strictly post-training on final checkpoint)

## 0. Install Pinned Dependencies

In [ ]:
!pip install -q peft==0.21.0 accelerate==1.15.0 transformers==4.49.0 datasets soundfile jiwer pandas pyyaml scipy

import os, sys, hashlib, torch, peft, transformers
from pathlib import Path

# --- Robust Repository & Application Root Discovery ---
candidate_roots = [
    Path.cwd(),
    Path.cwd() / 'vietnamese_asr_app',
    Path('/content/Vietnamese_ASR_Week6'),
    Path('/content/Vietnamese_ASR_Week6/vietnamese_asr_app'),
    Path('/content/AAIC'),
    Path('/content/AAIC/vietnamese_asr_app'),
]

app_root = None
for cand in candidate_roots:
    if (cand / 'scripts' / 'train_full_e1.py').exists() and (cand / 'manifests' / 'test_manifest.csv').exists():
        app_root = cand.resolve()
        break
    elif (cand / 'vietnamese_asr_app' / 'scripts' / 'train_full_e1.py').exists():
        app_root = (cand / 'vietnamese_asr_app').resolve()
        break

if app_root is None and Path('/content').exists():
    target_repo = Path('/content/Vietnamese_ASR_Week6')
    if not target_repo.exists():
        print(f'Cloning repository into {target_repo}...')
        get_ipython().system('git clone https://github.com/emt0147t/AAIC.git /content/Vietnamese_ASR_Week6')
    if (target_repo / 'vietnamese_asr_app' / 'scripts' / 'train_full_e1.py').exists():
        app_root = (target_repo / 'vietnamese_asr_app').resolve()

assert app_root is not None and (app_root / 'scripts' / 'train_full_e1.py').exists(), (
    f'CRITICAL ERROR: Could not locate canonical train script under {app_root}/scripts/train_full_e1.py. '
    'Please ensure the repository is cloned.'
)

# Set working directory to APP_ROOT
os.chdir(str(app_root))
if str(app_root) not in sys.path:
    sys.path.insert(0, str(app_root))

repo_root = app_root.parent if app_root.name == 'vietnamese_asr_app' else app_root
print(f'Repository Root: {repo_root}')
print(f'App Root (CWD):   {app_root}')
print(f'Train Script:     {(app_root / "scripts" / "train_full_e1.py").resolve()}')

print(f'OS:              {sys.platform}')
print(f'Python:          {sys.version.split()[0]}')
print(f'PyTorch:         {torch.__version__}')
print(f'PEFT:            {peft.__version__}')
print(f'Transformers:    {transformers.__version__}')
print(f'CUDA Available:  {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'CRITICAL ERROR: CUDA is required for FP16 Full E1 training!'

gpu_name = torch.cuda.get_device_name(0)
total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024**2)
print(f'GPU Model:       {gpu_name} ({total_vram_mb:.1f} MB)')

# Check frozen test manifest hash (Platform-independent canonical CRLF normalization)
test_manifest = "manifests/test_manifest.csv"
if os.path.exists(test_manifest):
    raw_bytes = open(test_manifest, "rb").read()
    raw_h = hashlib.sha256(raw_bytes).hexdigest()
    canonical_bytes = raw_bytes.replace(b"\r\n", b"\n").replace(b"\r", b"\n").replace(b"\n", b"\r\n")
    canonical_h = hashlib.sha256(canonical_bytes).hexdigest()
    print(f"Test Manifest RAW FILE SHA-256:         {raw_h}")
    print(f"Test Manifest CANONICAL FROZEN SHA-256: {canonical_h}")
    assert canonical_h == "efe54856d12613f109cebc9482a6f224ca4e2370d0ff3f8a2fe4ea0a013b60ca", "Test manifest altered!"
    print("Frozen test manifest integrity: PASS")


In [ ]:
import os, sys, hashlib, torch, peft, transformers

print(f"OS:              {sys.platform}")
print(f"Python:          {sys.version.split()[0]}")
print(f"PyTorch:         {torch.__version__}")
print(f"PEFT:            {peft.__version__}")
print(f"Transformers:    {transformers.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
assert torch.cuda.is_available(), 'CRITICAL ERROR: CUDA is required for FP16 Full E1 training!'

gpu_name = torch.cuda.get_device_name(0)
total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024**2)
print(f"GPU Model:       {gpu_name} ({total_vram_mb:.1f} MB)")

# Check frozen test manifest hash
test_manifest = 'manifests/test_manifest.csv'
if os.path.exists(test_manifest):
    h = hashlib.sha256(open(test_manifest, 'rb').read()).hexdigest()
    print(f"Test Hash:       {h}")
    assert h == 'efe54856d12613f109cebc9482a6f224ca4e2370d0ff3f8a2fe4ea0a013b60ca', 'Test manifest altered!'
    print("Test manifest integrity strictly VERIFIED.")

## 2. Load Model & Setup LoRA (cc51d32be916efebde04ff549854fa1741cb5c02)

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL_NAME = 'vinai/PhoWhisper-tiny'
MODEL_REVISION = 'cc51d32be916efebde04ff549854fa1741cb5c02'

processor = WhisperProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)

device = torch.device('cuda')
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none'
)
model = get_peft_model(base_model, lora_config)
model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100*trainable/total:.4f}%)")
assert trainable == 147456, f'Expected 147,456 trainable parameters, got {trainable}'

## 3. Verify Deterministic Exposure Policy Across Epochs

In [ ]:
import numpy as np

AUDITED_REPEATED_INDICES = {
    1: [186, 2394, 3596, 4922, 5311, 9527, 9922, 11904, 12434, 13590, 14261, 15356, 15589, 16068, 19817, 19918, 20082],
    2: [791, 2720, 3793, 4460, 4869, 6765, 7657, 8088, 10060, 10616, 12573, 15738, 17952, 22257, 23086, 23991, 26608],
    3: [329, 854, 868, 2852, 2949, 7126, 9800, 12278, 12298, 17346, 17498, 20781, 21165, 21287, 22885, 24648, 26633]
}

for ep in [1, 2, 3]:
    epoch_seed = 42 + (ep - 1)
    pad_rng = np.random.default_rng(epoch_seed + 1000)
    pad_idx = sorted(pad_rng.choice(26671, size=17, replace=False).tolist())
    assert pad_idx == AUDITED_REPEATED_INDICES[ep], f'Exposure policy mismatch in epoch {ep}!'
    print(f'Epoch {ep} exposure policy verified: matches audited schedule.')

# Launch full training with FP16 GradScaler, saving to checkpoints/FULL_E1/
print('Starting Full E1 Training Execution...')
canonical_script = os.path.join(str(app_root), 'scripts', 'train_full_e1.py')
print(f'Canonical Script: {canonical_script}')
!python "{canonical_script}"


In [ ]:
# Launch full training with FP16 GradScaler, saving to checkpoints/FULL_E1/
print("Starting Full E1 Training Execution...")
!python scripts/train_full_e1.py